In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import csv
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from functools import reduce

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
# little function to define the file root on different machines
def find_f_root(start_path: Path = Path.cwd(), anchor: str = "CASA0004_work") -> Path:
    """
    Traverse up from the start_path until the anchor folder is found. Returns the path to the anchor folder.
    """
    for parent in [start_path] + list(start_path.parents):
        if parent.name == anchor:
            return parent
    raise FileNotFoundError(f"Anchor folder '{anchor}' not found in path hierarchy.")
  
f_root = find_f_root()

In [ ]:
def read_nomis_sheets(file_path, sheet_config, age_groups, merge_col,
                      skiprows=9, nrows=None):

    dfs = {}

    for cfg in sheet_config:
        df = pd.read_excel(
            file_path,
            sheet_name=cfg["sheet"],
            skiprows=skiprows,
            nrows=nrows,
            usecols=cfg["usecols"]
        )

        required = [merge_col, *age_groups]
        missing = [c for c in required if c not in df.columns]

        if missing:
            raise KeyError(
                f"Sheet {cfg['sheet']} missing {missing}. "
                f"Available: {df.columns.tolist()}"
            )

        df = df[required].copy()
        df[list(age_groups)] = df[list(age_groups)].apply(
            pd.to_numeric, errors="coerce"
        )

        out = pd.DataFrame({merge_col: df[merge_col]})

        for age in dict.fromkeys(age_groups.values()):
            cols = [c for c, target in age_groups.items() if target == age]
            out[f"{age}{cfg['suffix']}"] = df[cols].sum(axis=1, min_count=1)

        dfs[cfg["label"]] = out

    return dfs

# 2021 Census

In [ ]:
# --------------------------------------------------
# Configuration
# --------------------------------------------------
file_path = (f_root / "data/census/2021/ltla_mrp_tables" / "nomis_2026_07_31_193558.xlsx")
merge_col = "Unnamed: 1"
skiprows = 9
nrows = 318
age_groups = {
    "Aged 16 to 24 years": "age_16_24",
    "Aged 25 to 34 years": "age_25_34",
    "Aged 35 to 49 years": "age_35_64",
    "Aged 50 to 64 years": "age_35_64",
    "Aged 65 years and over": "age_65_plus",
}
# One entry per worksheet
sheet_config = [
    {
        "sheet": 0, "label": "employed_excl_students", "suffix": "_Employed", "usecols": range(0, 7),
    },
    {
        "sheet": 1, "label": "unemployed_excl_students", "suffix": "_Unemployed_or_economically_inactive", "usecols": range(0, 7),
    },
    {
        "sheet": 2, "label": "employed_students", "suffix": "_Employed", "usecols": range(0, 7),
    },
    {
        "sheet": 3, "label": "unemployed_students", "suffix": "_Unemployed_or_economically_inactive", "usecols": range(0, 7),
    },
    {
        "sheet": 4, "label": "inactive_excl_students", "suffix": "_Unemployed_or_economically_inactive", "usecols": range(0, 7),
    },
    {
        "sheet": 5, "label": "inactive_students", "suffix": "_Unemployed_or_economically_inactive", "usecols": range(0, 7),
    },
]

nomis_dfs = read_nomis_sheets(
    file_path=file_path,
    sheet_config=sheet_config,
    age_groups=age_groups,
    merge_col=merge_col,
    skiprows=skiprows,
    nrows=nrows
)

employed_excl_students_df_2021 = nomis_dfs["employed_excl_students"]
employed_students_df_2021 = nomis_dfs["employed_students"]
employed_df_2021 = (pd.concat([employed_excl_students_df_2021, employed_students_df_2021]).groupby("Unnamed: 1", as_index=False).sum(numeric_only=True))

unemployed_excl_students_df_2021 = nomis_dfs["unemployed_excl_students"]
unemployed_students_df_2021 = nomis_dfs["unemployed_students"]
inactive_excl_students_df_2021 = nomis_dfs["inactive_excl_students"]
inactive_students_df_2021 = nomis_dfs["inactive_students"]
unemployed_df_2021 = (
    pd.concat([
        unemployed_excl_students_df_2021,
        unemployed_students_df_2021,
        inactive_excl_students_df_2021,
        inactive_students_df_2021,
    ], ignore_index=True)
    .groupby("Unnamed: 1", as_index=False)
    .sum(numeric_only=True)
)
#making the 2021 df
source_2021 = (
    employed_df_2021
    .merge(unemployed_df_2021, left_on="Unnamed: 1", right_on="Unnamed: 1", how="left")
)
source_2021

# 2011 Census

In [ ]:
# --------------------------------------------------
# Configuration
# --------------------------------------------------
file_path = (f_root / "data/census/2011/ltls_mrp_tables/nomis_2026_07_31_192700.xlsx")
merge_col = "Unnamed: 1"
skiprows = 9
nrows = 348
age_groups = {
    "Age 16 to 24": "age_16_24",
    "Age 25 to 34": "age_25_34",
    "Age 35 to 49": "age_35_64",
    "Age 50 to 64": "age_35_64",
    "Age 65 and over": "age_65_plus"

}
# One entry per worksheet
sheet_config = [
    {
        "sheet": 0, "label": "employed", "suffix": "_Employed", "usecols": range(0, 7),
    },
    {
        "sheet": 1, "label": "unemployed", "suffix": "_Unemployed_or_economically_inactive", "usecols": range(0, 7),
    },
        {
        "sheet": 2, "label": "inactive", "suffix": "_Unemployed_or_economically_inactive", "usecols": range(0, 7),
    },
]

nomis_dfs = read_nomis_sheets(
    file_path=file_path,
    sheet_config=sheet_config,
    age_groups=age_groups,
    merge_col=merge_col,
    skiprows=skiprows,
    nrows=nrows
)

employed_df_2011 = nomis_dfs["employed"]
unemployed_df_2011 = nomis_dfs["unemployed"]
inactive_df_2011 = nomis_dfs["inactive"]
unemployed_inactive_df_2011 = (pd.concat([unemployed_df_2011, inactive_df_2011]).groupby("Unnamed: 1", as_index=False).sum(numeric_only=True))

#making the 201 df
source_2011 = (
    employed_df_2011
    .merge(unemployed_inactive_df_2011, left_on="Unnamed: 1", right_on="Unnamed: 1", how="left")
)
source_2011

In [ ]:
lookup = pd.read_csv(f_root / "data/geographies/ltla/2014_to_2023_lad_lookup.csv")
lookup

In [ ]:
source_2011_lookup = (
    source_2011
    .merge(lookup, left_on="Unnamed: 1", right_on="LAD14CD", how="left")
)

In [ ]:
source_2011_lookup = (
    source_2011_lookup.drop(columns=["Unnamed: 1", "LAD14CD"])
)

In [ ]:
source_2011_lookup = (
    source_2011_lookup
    .groupby("LAD23CD", as_index=False)
    .sum(numeric_only=True)
)

In [ ]:
source_2011_lookup

In [ ]:
combined = (
    pd.concat([
        source_2021.loc[:, ~source_2021.columns.duplicated()]
            .rename(columns={"Unnamed: 1": "LAD23CD"}),
        source_2011_lookup.loc[:, ~source_2011_lookup.columns.duplicated()]
    ], ignore_index=True)
    .groupby("LAD23CD", as_index=False)
    .mean(numeric_only=True)
)

combined

In [ ]:
combined.columns

In [ ]:
small_geogr = gpd.read_file(f_root / "data/geographies/ltla/export_small_geogr_ew.shp") #lower-tier local authorities, BSC | district/unitary
small_geogr = (
    small_geogr
    .merge(combined, left_on="LAD23CD", right_on="LAD23CD", how="left")
)
small_geogr.info()

Amending the numbers by GOR to match the % of victimised population.

In [ ]:
victim_lookup = pd.read_csv(f_root / "data/csew/exp_for_inla_stage/perc_pop_victim.csv")
victim_lookup

In [ ]:
post_strat_vars = ["agelong", "remploya", "gor"]
geo_col = "gor"
cell_col = "_".join(post_strat_vars)

lookup_rate_col = "share_pop_victim"
population_col = "population_women"
output_col = "victim_population"
wide_cell_col = "poststrat_cell"
row_id_col = "_row_id"

id_cols = ["LAD23CD", "LAD23NM", "geometry", geo_col]

poststrat_cols = [
    c for c in small_geogr.columns
    if c not in id_cols
]

#from wide to long format
long = (
    small_geogr
    .assign(**{row_id_col: np.arange(len(small_geogr))})
    .melt(
        id_vars=[row_id_col, *id_cols],
        value_vars=poststrat_cols,
        var_name=wide_cell_col,
        value_name=population_col
    )
)

long[cell_col] = (
    long[wide_cell_col].astype("string").str.strip()
    .str.cat(long[geo_col].astype("string").str.strip(), sep="_")
)

long[cell_col] = (
    long[wide_cell_col]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.cat(
        long[geo_col].astype("string").str.strip(),
        sep="_"
    )
)

long = long.merge(
    victim_lookup[[cell_col, lookup_rate_col]],
    on=cell_col, how="left", validate="many_to_one")

# Display combinations with no matching
unmatched = (
    long.loc[
        long[lookup_rate_col].isna(),
        [wide_cell_col, geo_col, cell_col]
    ].drop_duplicates())

print("Unmatched lookup combinations:")
display(unmatched)

# Apply weighted prevalence to population counts
long[output_col] = (
    pd.to_numeric(long[population_col], errors="coerce")
    * pd.to_numeric(long[lookup_rate_col], errors="coerce")
)

# Return estimated victim populations to wide format
victim_wide = (
    long
    .pivot(
        index=row_id_col,
        columns=wide_cell_col,
        values=output_col
    )
    .reset_index()
    .rename_axis(columns=None)
)

# Replace population columns with estimated victim-population columns
non_poststrat_cols = [
    col for col in small_geogr.columns
    if col not in poststrat_cols
]

small_geogr = (
    small_geogr
    .assign(**{row_id_col: np.arange(len(small_geogr))})
    .drop(columns=poststrat_cols)
    .merge(
        victim_wide,
        on=row_id_col,
        how="left",
        validate="one_to_one"
    )
    .drop(columns=row_id_col)
    [non_poststrat_cols + poststrat_cols]
)

In [ ]:
small_geogr

In [ ]:
small_geogr.columns

In [ ]:
small_geogr.to_file(
    f_root / "data/geographies/ltla/export_small_geogr_ew_with_covariates_1.gpkg",
    driver="GPKG"
)